# CardioAI — Intelligent Heart Disease Prediction System
### 4th Semester | AI & ML Combined Project | 2026

**Project Overview:**  
This notebook implements a complete AI + ML clinical decision support system for heart disease prediction.  
It integrates five components in a single pipeline:

| # | Component | Technique |
|---|-----------|-----------|
| 1 | Risk Prediction | XGBoost (Supervised ML) |
| 2 | Care Pathway Planning | A* Search (AI Planning) |
| 3 | Clinical Reasoning | Forward Chaining (Knowledge Base) |
| 4 | Medical Text Search | TF-IDF + Cosine Similarity (NLP) |
| 5 | PEAS Agent Definition | Formal AI Agent Framework |

**Dataset:** UCI Heart Disease Dataset — 920 patients across 4 hospitals  
**Model Performance:** AUC 0.898 | Recall 0.931 | Threshold 0.35


---
## Section 1 — Setup & Model Loading
> Import libraries and load the pre-trained XGBoost model and StandardScaler from Google Drive.

In [ ]:
import joblib
import pandas as pd
import heapq
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Model & Scaler Paths ──────────────────────────────────────────────────────
MODEL_PATH  = "/content/drive/MyDrive/AI-ML/heart_disease_model.pkl"
SCALER_PATH = "/content/drive/MyDrive/AI-ML/heart_scaler.pkl"

# ── Load Artifacts ────────────────────────────────────────────────────────────
try:
    model  = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    print("✅ Model and Scaler loaded successfully.")
except FileNotFoundError:
    print("❌ File not found. Mount Google Drive and check paths.")
    model, scaler = None, None
except Exception as e:
    print(f"❌ Error: {e}")
    model, scaler = None, None

---
## Section 2 — PEAS Framework (Formal Agent Definition)
> Before building the AI agent, we formally define it using the standard PEAS framework.

In [ ]:
# ── PEAS Framework — CardioAI Intelligent Agent ──────────────────────────────
# PEAS = Performance Measure, Environment, Actuators, Sensors
# This is the standard formal specification for any AI agent.

peas = {
    "Performance Measure": [
        "Correctly classify patient risk: High / Medium / Low",
        "Minimise false negatives — missed disease is life-threatening",
        "Find optimal care pathway in minimum cost steps (A* optimality)",
        "Maximise recall: XGBoost achieves 0.931 at threshold 0.35",
        "Produce clinically actionable recommendations grounded in guidelines",
    ],
    "Environment": [
        "Hospital outpatient clinic / GP surgery / cardiology unit",
        "Patient with 14 clinical measurements (UCI Heart Disease dataset)",
        "Partially observable — agent sees measurements, not ground truth",
        "Deterministic — same inputs always produce same outputs",
        "Static — environment does not change during a single assessment",
        "Discrete — risk states are categorical (High / Medium / Low)",
    ],
    "Actuators": [
        "Output risk classification (High / Medium / Low Risk)",
        "Trigger A* search → optimal clinical intervention pathway",
        "Recommend Blood Pressure Management (if trestbps >= 140)",
        "Recommend Cholesterol Management (if chol >= 240)",
        "Order ECG Evaluation (if exercise-induced angina present)",
        "Order Stress Testing (if oldpeak > 2.0 mm)",
        "Refer to Cardiology Consultation (if age >= 60 or High Risk)",
        "Activate Knowledge Base for forward chaining inference",
        "Return TF-IDF matched medical knowledge for clinical context",
    ],
    "Sensors": [
        "age         — patient age in years (28–77)",
        "sex         — biological sex (0=Female, 1=Male)",
        "cp          — chest pain type (4 categories)",
        "trestbps    — resting blood pressure in mmHg",
        "chol        — serum cholesterol in mg/dl",
        "fbs         — fasting blood sugar > 120 mg/dl (boolean)",
        "restecg     — resting ECG results (3 categories)",
        "thalch      — maximum heart rate achieved (bpm)",
        "exang       — exercise-induced angina (boolean)",
        "oldpeak     — ST depression induced by exercise (mm)",
        "slope       — slope of peak exercise ST segment",
        "thal        — thalassemia type (normal/fixed/reversible)",
        "ml_prob     — XGBoost predicted disease probability (0.0–1.0)",
    ],
}

print("=" * 60)
print("  PEAS FRAMEWORK — CardioAI Intelligent Agent")
print("=" * 60)
for component, items in peas.items():
    print(f"\n  📌 {component}:")
    for item in items:
        print(f"     • {item}")

print("\n" + "=" * 60)
print("  AGENT TYPE: Goal-based + Utility-based + Knowledge-based")
print("=" * 60)
print("  → Goal-based   : A* finds optimal path to care goal state")
print("  → Utility-based: Threshold 0.35 maximises recall (not 0.50)")
print("  → KB-based     : Forward chaining derives clinical conclusions")
print("\n✅ PEAS Framework defined!")

---
## Section 3 — Feature Engineering
> Convert raw patient inputs into the exact feature format the model was trained on.  
> This includes one-hot encoding of categorical variables and creation of derived features.

**Patient values used below are example defaults — replace with real patient data.**

In [ ]:
# ── Default Patient Values (replace with real inputs) ────────────────────────
age          = 63
sex          = 1       # 1=Male, 0=Female
cp_choice    = 4       # 1=Typical Angina, 2=Atypical, 3=Non-anginal, 4=Asymptomatic
trestbps     = 145     # Resting blood pressure (mmHg)
chol         = 233     # Serum cholesterol (mg/dl)
fbs          = 1       # Fasting blood sugar > 120? (1=Yes, 0=No)
thalch       = 150     # Max heart rate achieved
exang        = 0       # Exercise-induced angina (1=Yes, 0=No)
oldpeak      = 2.3     # ST depression (mm)
slope_choice = 2       # 1=Upsloping, 2=Flat, 3=Downsloping
thal_choice  = 3       # 1=Normal, 2=Fixed Defect, 3=Reversible Defect
restecg_choice = 0     # 0=Normal, 1=ST-T Abnormality, 2=LV Hypertrophy

# ── One-Hot Encoding: Chest Pain Type ────────────────────────────────────────
cp_typical_angina  = 1 if cp_choice == 1 else 0
cp_atypical_angina = 1 if cp_choice == 2 else 0
cp_non_anginal     = 1 if cp_choice == 3 else 0
# cp_choice == 4 (Asymptomatic) → all three above = 0 (reference category)

# ── One-Hot Encoding: Resting ECG ────────────────────────────────────────────
restecg_normal        = 1 if restecg_choice == 1 else 0
restecg_st_abnormality = 1 if restecg_choice == 2 else 0

# ── One-Hot Encoding: Slope ───────────────────────────────────────────────────
slope_upsloping = 1 if slope_choice == 1 else 0
slope_flat      = 1 if slope_choice == 2 else 0

# ── One-Hot Encoding: Thalassemia ────────────────────────────────────────────
thal_normal          = 1 if thal_choice == 1 else 0
thal_fixed_defect    = 1 if thal_choice == 2 else 0
thal_reversable_defect = 1 if thal_choice == 3 else 0

# ── Derived Features ─────────────────────────────────────────────────────────
def age_risk_group(a):
    if a < 40:   return 0
    elif a < 55: return 1
    elif a < 65: return 2
    else:        return 3

age_risk        = age_risk_group(age)
bp_chol_interaction  = trestbps * chol
exercise_stress_score = exang + oldpeak
predicted_max_hr = 220 - age
thalch_age_ratio = thalch / predicted_max_hr if predicted_max_hr != 0 else 0

# ── Assemble Patient DataFrame ────────────────────────────────────────────────
patient_df = pd.DataFrame([{
    'age':                      age,
    'sex':                      sex,
    'trestbps':                 trestbps,
    'chol':                     chol,
    'fbs':                      fbs,
    'thalch':                   thalch,
    'exang':                    exang,
    'oldpeak':                  oldpeak,
    'cp_atypical angina':       cp_atypical_angina,
    'cp_non-anginal':           cp_non_anginal,
    'cp_typical angina':        cp_typical_angina,
    'restecg_normal':           restecg_normal,
    'restecg_st-t abnormality': restecg_st_abnormality,
    'slope_flat':               slope_flat,
    'slope_upsloping':          slope_upsloping,
    'thal_normal':              thal_normal,
    'thal_reversable defect':   thal_reversable_defect,
    'age_risk_group':           age_risk,
    'bp_chol_interaction':      bp_chol_interaction,
    'exercise_stress_score':    exercise_stress_score,
    'thalch_age_ratio':         thalch_age_ratio,
}])

print("✅ Patient DataFrame created — shape:", patient_df.shape)
display(patient_df.T.rename(columns={0: "Value"}))

---
## Section 4 — ML Prediction & Risk Assessment
> Scale patient data and run through the XGBoost model to get a probability and risk state.

**Threshold = 0.35** (not 0.50) — chosen to maximise recall and catch all disease cases.

In [ ]:
# ── Scale & Predict ───────────────────────────────────────────────────────────
if model and scaler:
    patient_scaled = scaler.transform(patient_df)
    prediction     = model.predict(patient_scaled)[0]
    probability    = model.predict_proba(patient_scaled)[0][1]
else:
    # Fallback demo values if model not loaded
    print("⚠️  Model not loaded — using demo values.")
    prediction, probability = 1, 0.82

# ── Risk State Classification ─────────────────────────────────────────────────
def determine_risk(prob):
    if prob >= 0.80:   return "High Risk"
    elif prob >= 0.40: return "Medium Risk"
    else:              return "Low Risk"

risk_state = determine_risk(probability)

# ── Display Results ───────────────────────────────────────────────────────────
print("=" * 50)
print("  ML PREDICTION RESULTS")
print("=" * 50)
print(f"  Prediction  : {'Heart Disease' if prediction == 1 else 'No Heart Disease'}")
print(f"  Probability : {round(probability * 100, 2)}%")
print(f"  Risk State  : {risk_state}")
print(f"  Threshold   : 0.35  (recall-optimised)")
print("=" * 50)

---
## Section 5 — AI Planning: A* Search
> Use A* search to find the optimal (minimum cost) clinical care pathway.  
> The ML risk state becomes the **initial state**; the goal is "Goal State".

**Formal Problem Tuple: P = (S, s₀, A, T, C, G, h)**

In [ ]:
# ── State Space Definition ────────────────────────────────────────────────────
print("=" * 60)
print("  FORMAL STATE SPACE")
print("=" * 60)
print("\n  Initial States (set by ML model output):")
print("    [High Risk]   → XGBoost prob >= 0.80 — urgent intervention")
print("    [Medium Risk] → XGBoost prob 0.40–0.80 — close monitoring")
print("    [Low Risk]    → XGBoost prob < 0.40 — routine care")
print("\n  Intermediate States (interventions):")
for s in ["Blood Pressure Management","Cholesterol Management",
          "ECG Evaluation","Stress Evaluation",
          "Cardiology Consultation","Risk Stratification","Treatment Planning"]:
    print(f"    [{s}]")
print("\n  Goal State:")
print("    [Goal State] → Patient has received optimal care pathway")
print("\n  Action cost = clinical resource intensity (1=low, 2=med, 3=high)")

In [ ]:
# ── Build Dynamic Medical Graph ───────────────────────────────────────────────
# Graph edges are built based on THIS patient's clinical values.
medical_graph = {risk_state: {}}

# Evidence-based rules for initial transitions
if trestbps >= 140:  medical_graph[risk_state]["Blood Pressure Management"] = 2
if chol >= 240:      medical_graph[risk_state]["Cholesterol Management"]    = 2
if exang == 1:       medical_graph[risk_state]["ECG Evaluation"]            = 1
if oldpeak > 2:      medical_graph[risk_state]["Stress Evaluation"]         = 1
if age >= 60:        medical_graph[risk_state]["Cardiology Consultation"]   = 1

# If no specific conditions, add a default pathway
if not medical_graph[risk_state]:
    medical_graph[risk_state]["Cardiology Consultation"] = 1

# Standard intermediate transitions
medical_graph["Blood Pressure Management"] = {"Cardiology Consultation": 1}
medical_graph["Cholesterol Management"]    = {"Cardiology Consultation": 1}
medical_graph["ECG Evaluation"]            = {"Cardiology Consultation": 1}
medical_graph["Stress Evaluation"]         = {"Cardiology Consultation": 1}
medical_graph["Cardiology Consultation"]   = {"Risk Stratification": 1}
medical_graph["Risk Stratification"]       = {"Treatment Planning": 1}
medical_graph["Treatment Planning"]        = {"Goal State": 0}

print("✅ Medical graph built. Edges from initial state:")
for neighbor, cost in medical_graph[risk_state].items():
    print(f"   {risk_state} → {neighbor}  (cost={cost})")

In [ ]:
# ── Admissible Heuristic ─────────────────────────────────────────────────────
# h(n) = estimated remaining cost to Goal State
# Admissible: never overestimates → guarantees optimal path
heuristic = {
    "High Risk":                 5,
    "Medium Risk":               3,
    "Low Risk":                  1,
    "Blood Pressure Management": 4,
    "Cholesterol Management":    4,
    "ECG Evaluation":            3,
    "Stress Evaluation":         3,
    "Cardiology Consultation":   2,
    "Risk Stratification":       1,
    "Treatment Planning":        1,
    "Goal State":                0,
}
print("✅ Heuristic defined — admissible (never overestimates).")

In [ ]:
# ── A* Search Algorithm ──────────────────────────────────────────────────────
def a_star_search(graph, start, goal, heuristic):
    """
    A* Search: finds the minimum-cost path from start to goal.
    f(n) = g(n) + h(n)  where g = actual cost, h = heuristic estimate
    Returns: (path, total_cost)
    """
    queue   = [(0, start, [start], 0)]   # (f, node, path, g)
    visited = set()

    while queue:
        f, node, path, g = heapq.heappop(queue)

        if node == goal:
            return path, g

        if node in visited:
            continue
        visited.add(node)

        for neighbor, cost in graph.get(node, {}).items():
            new_g = g + cost
            new_f = new_g + heuristic.get(neighbor, 0)
            heapq.heappush(queue, (new_f, neighbor, path + [neighbor], new_g))

    return None, None

print("✅ A* search algorithm defined.")

In [ ]:
# ── Execute A* Search ─────────────────────────────────────────────────────────
path, cost = a_star_search(medical_graph, risk_state, "Goal State", heuristic)

print("=" * 60)
print("  A* SEARCH — OPTIMAL CARE PATHWAY")
print("=" * 60)
print(f"  Start : {risk_state}")
print(f"  Goal  : Goal State")
print()
if path:
    for i, state in enumerate(path):
        prefix = "  START →" if i == 0 else ("  GOAL  " if i == len(path)-1 else "         →")
        print(f"{prefix} {state}")
    print(f"\n  Total Path Cost : {cost}")
    print(f"  Steps           : {len(path)}")
else:
    print("  ❌ No path found.")

In [ ]:
# ── A* Search Trace (Explainability) ─────────────────────────────────────────
def a_star_trace(graph, start, goal, heuristic):
    """Same as a_star_search but prints each exploration step for transparency."""    queue   = [(0, start, [start], 0)]
    visited = set()
    print("  SEARCH TRACE")
    print("  " + "-" * 50)

    while queue:
        f, node, path, g = heapq.heappop(queue)
        print(f"  Exploring: [{node}]  f={f}, g={g}, h={heuristic.get(node,0)}")

        if node == goal:
            print("\n  ✅ Goal Reached!")
            return path, g
        if node in visited:
            continue
        visited.add(node)

        for neighbor, cost in graph.get(node, {}).items():
            new_g = g + cost
            new_f = new_g + heuristic.get(neighbor, 0)
            print(f"    → Action: {neighbor}  | g={new_g}, h={heuristic.get(neighbor,0)}, f={new_f}")
            heapq.heappush(queue, (new_f, neighbor, path + [neighbor], new_g))

    return None, None

print("=" * 60)
print("  A* SEARCH TRACE")
print("=" * 60)
a_star_trace(medical_graph, risk_state, "Goal State", heuristic);

In [ ]:
# ── State Space Graph Visualisation ──────────────────────────────────────────
G = nx.DiGraph()
for node, neighbors in medical_graph.items():
    for neighbor, cost in neighbors.items():
        G.add_edge(node, neighbor, weight=cost)

plt.figure(figsize=(14, 8))
pos = nx.spring_layout(G, seed=42, k=1.2)

node_colors = []
for n in G.nodes():
    if n == risk_state:         node_colors.append("#FF6B6B")
    elif n == "Goal State":     node_colors.append("#51CF66")
    else:                       node_colors.append("#74C0FC")

nx.draw(G, pos, with_labels=True, node_size=3500,
        node_color=node_colors, font_size=9, font_weight="bold",
        edge_color="#555", arrows=True, arrowsize=20)
nx.draw_networkx_edge_labels(G, pos,
    edge_labels=nx.get_edge_attributes(G, "weight"),
    font_color="darkred", font_size=9)

plt.title("CardioAI — Medical Planning State Space (A* Graph)", size=14, weight="bold")
plt.legend(handles=[
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#FF6B6B', markersize=12, label='Initial State'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#74C0FC', markersize=12, label='Intermediate State'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#51CF66', markersize=12, label='Goal State'),
], loc="upper right")
plt.tight_layout()
plt.show()

---
## Section 6 — Knowledge-Based Expert System (Forward Chaining)
> Apply 24 IF-THEN medical rules to infer clinical conclusions from patient facts.  
> Cross-validates the ML prediction and A* pathway.

**Rule Tiers:**
- **Tier 1** — Single-feature risk flags (R01–R09)
- **Tier 2** — Combined risk rules (R10–R14)  
- **Tier 3** — Clinical pathway rules (R15–R24)

In [ ]:
# ── Generate Initial Facts from Patient Data + ML Output ─────────────────────
def generate_initial_facts(age, trestbps, chol, exang, oldpeak, risk_state,
                            thalch=150, thal_choice=1, restecg_choice=0, cp_choice=4):
    facts = set()

    # Risk state from ML model (bridges ML → KB)
    if risk_state == "High Risk":   facts.add("high_risk")
    elif risk_state == "Medium Risk": facts.add("medium_risk")
    else:                            facts.add("low_risk")

    if age >= 60:             facts.add("elderly_patient")
    if trestbps >= 140:       facts.add("high_bp")
    if chol >= 240:           facts.add("high_cholesterol")
    if thalch < 120:          facts.add("low_max_hr")
    if exang == 1:            facts.add("exercise_angina")
    if oldpeak > 2:           facts.add("high_oldpeak")
    if thal_choice == 3:      facts.add("reversible_thal")
    if restecg_choice in [1,2]: facts.add("ecg_abnormal")
    if cp_choice == 4:        facts.add("silent_ischemia_risk")

    return facts

initial_facts = generate_initial_facts(
    age, trestbps, chol, exang, oldpeak, risk_state,
    thalch, thal_choice, restecg_choice, cp_choice
)

print("INITIAL FACTS:")
print("=" * 50)
for f in sorted(initial_facts):
    print(f"  ✓ {f}")
print(f"\nTotal: {len(initial_facts)} facts")

In [ ]:
# ── Knowledge Base: 24 IF-THEN Rules ─────────────────────────────────────────
knowledge_base = [
    # ── TIER 1: Single-Feature Risk Flags ────────────────────────────────────
    {"id":"R01", "if":["elderly_patient"],    "then":"age_related_cardiac_risk",
     "source":"AHA — age >60 doubles baseline cardiac risk"},
    {"id":"R02", "if":["high_bp"],             "then":"hypertension",
     "source":"AHA/ACC — systolic BP >= 140 = Stage 2 hypertension"},
    {"id":"R03", "if":["high_cholesterol"],    "then":"hyperlipidemia",
     "source":"NHLBI — total cholesterol >= 240 mg/dl = high risk"},
    {"id":"R04", "if":["low_max_hr"],          "then":"reduced_cardiac_reserve",
     "source":"Mayo Clinic — max HR <120 bpm indicates poor cardiac reserve"},
    {"id":"R05", "if":["exercise_angina"],     "then":"possible_ischemia",
     "source":"Mayo Clinic/NHLBI — exercise angina indicates myocardial ischemia"},
    {"id":"R06", "if":["high_oldpeak"],        "then":"abnormal_stress_response",
     "source":"Cardiology — ST depression >2mm = significant ischemia"},
    {"id":"R07", "if":["reversible_thal"],     "then":"reversible_perfusion_defect",
     "source":"Nuclear cardiology — reversible defect = stress-induced ischemia"},
    {"id":"R08", "if":["ecg_abnormal"],        "then":"ecg_detected_abnormality",
     "source":"AHA — ST-T wave abnormality on resting ECG = cardiac risk marker"},
    {"id":"R09", "if":["silent_ischemia_risk"],"then":"asymptomatic_cp",
     "source":"Dataset insight — asymptomatic cp has highest disease prevalence"},

    # ── TIER 2: Combined Risk Rules ───────────────────────────────────────────
    {"id":"R10", "if":["hypertension","hyperlipidemia"],
     "then":"elevated_cardiovascular_risk",
     "source":"AHA — combined hypertension + hyperlipidemia = metabolic syndrome"},
    {"id":"R11", "if":["possible_ischemia","high_risk"],
     "then":"suspected_coronary_artery_disease",
     "source":"ACC — ischemia + high ML risk = probable CAD"},
    {"id":"R12", "if":["abnormal_stress_response","possible_ischemia"],
     "then":"requires_ecg",
     "source":"Mayo Clinic — ST depression + angina mandates ECG evaluation"},
    {"id":"R13", "if":["elderly_patient","silent_ischemia_risk"],
     "then":"critical_screening_needed",
     "source":"AHA — elderly + asymptomatic = highest missed-diagnosis risk"},
    {"id":"R14", "if":["reduced_cardiac_reserve","reversible_perfusion_defect"],
     "then":"exercise_cardiac_failure_risk",
     "source":"Cardiology — low max HR + reversible defect = cardiac stress failure"},

    # ── TIER 3: Clinical Pathway Rules ────────────────────────────────────────
    {"id":"R15", "if":["requires_ecg"],
     "then":"diagnostic_testing",
     "source":"Clinical pathway — ECG requirement triggers full diagnostic workup"},
    {"id":"R16", "if":["diagnostic_testing"],
     "then":"cardiology_consultation",
     "source":"Clinical pathway — diagnostic testing requires specialist review"},
    {"id":"R17", "if":["high_risk"],
     "then":"close_monitoring",
     "source":"AHA — high-risk patients require intensive monitoring protocol"},
    {"id":"R18", "if":["close_monitoring","diagnostic_testing"],
     "then":"specialist_followup",
     "source":"Clinical pathway — monitoring + diagnostics → specialist follow-up"},
    {"id":"R19", "if":["specialist_followup"],
     "then":"treatment_planning",
     "source":"Clinical pathway — specialist review leads to treatment plan"},
    {"id":"R20", "if":["low_risk"],
     "then":"preventive_education",
     "source":"WHO — low-risk patients benefit from preventive health education"},
    {"id":"R21", "if":["preventive_education"],
     "then":"healthy_lifestyle",
     "source":"WHO — promoting healthy lifestyle for low-risk individuals"},
    {"id":"R22", "if":["healthy_lifestyle"],
     "then":"routine_monitoring",
     "source":"Clinical pathway — routine monitoring for stable low-risk patients"},
    {"id":"R23", "if":["medium_risk"],
     "then":"followup_assessment",
     "source":"AHA — medium-risk patients require re-evaluation and follow-up"},
    {"id":"R24", "if":["followup_assessment"],
     "then":"lifestyle_counseling",
     "source":"Clinical pathway — tailored lifestyle advice for medium-risk patients"},
]

print(f"✅ Knowledge Base loaded: {len(knowledge_base)} rules across 3 tiers.")

In [ ]:
# ── Forward Chaining Engine ───────────────────────────────────────────────────
def forward_chaining(facts, rules):
    """
    Forward Chaining Inference:
    Start from known facts, apply IF-THEN rules repeatedly
    until no new facts can be derived (fixed point).
    Returns: (all_facts, trace_of_fired_rules)
    """
    inferred = set(facts)
    trace    = []
    changed  = True

    while changed:
        changed = False
        for rule in rules:
            conditions = set(rule["if"])
            conclusion = rule["then"]
            if conditions.issubset(inferred) and conclusion not in inferred:
                inferred.add(conclusion)
                trace.append({"rule": rule["id"], "conditions": list(conditions),
                               "conclusion": conclusion, "source": rule["source"]})
                changed = True

    return inferred, trace

# ── Run Inference ─────────────────────────────────────────────────────────────
final_facts, inference_trace = forward_chaining(initial_facts, knowledge_base)

print("=" * 60)
print("  FORWARD CHAINING — INFERENCE TRACE")
print("=" * 60)
if inference_trace:
    for entry in inference_trace:
        print(f"  [{entry['rule']}] {entry['conditions']} → {entry['conclusion']}")
        print(f"         Source: {entry['source']}")
else:
    print("  No additional rules fired.")

new_facts = final_facts - initial_facts
print(f"\n  Rules fired    : {len(inference_trace)}")
print(f"  New conclusions: {len(new_facts)}")
print("\n" + "=" * 60)
print("  FINAL INFERRED CONCLUSIONS")
print("=" * 60)
for fact in sorted(new_facts):
    print(f"  ✓ {fact}")

---
## Section 7 — Contrasting Cases: High Risk vs Low Risk
> Demonstrate that different ML outputs lead to different A* paths and KB conclusions.  
> Required by the assignment: "Run on at least two contrasting examples."

In [ ]:
# ── Case A: High Risk Patient ─────────────────────────────────────────────────
case_a = {
    'age':63, 'trestbps':158, 'chol':275, 'exang':1,
    'oldpeak':3.2, 'risk_state':'High Risk',
    'thalch':105, 'thal_choice':3, 'restecg_choice':1, 'cp_choice':4
}

# A* for Case A
graph_a = {'High Risk': {}}
if case_a['trestbps'] >= 140: graph_a['High Risk']['Blood Pressure Management'] = 2
if case_a['chol'] >= 240:     graph_a['High Risk']['Cholesterol Management']    = 2
if case_a['exang'] == 1:      graph_a['High Risk']['ECG Evaluation']            = 1
if case_a['oldpeak'] > 2:     graph_a['High Risk']['Stress Evaluation']         = 1
if case_a['age'] >= 60:       graph_a['High Risk']['Cardiology Consultation']   = 1
graph_a.update({
    'Blood Pressure Management': {'Cardiology Consultation': 1},
    'Cholesterol Management':    {'Cardiology Consultation': 1},
    'ECG Evaluation':            {'Cardiology Consultation': 1},
    'Stress Evaluation':         {'Cardiology Consultation': 1},
    'Cardiology Consultation':   {'Risk Stratification': 1},
    'Risk Stratification':       {'Treatment Planning': 1},
    'Treatment Planning':        {'Goal State': 0},
})
path_a, cost_a = a_star_search(graph_a, 'High Risk', 'Goal State', heuristic)
facts_a  = generate_initial_facts(**{k:v for k,v in case_a.items()})
final_a, trace_a = forward_chaining(facts_a, knowledge_base)

# ── Case B: Low Risk Patient ──────────────────────────────────────────────────
case_b = {
    'age':38, 'trestbps':112, 'chol':185, 'exang':0,
    'oldpeak':0.1, 'risk_state':'Low Risk',
    'thalch':178, 'thal_choice':1, 'restecg_choice':0, 'cp_choice':2
}

graph_b = {'Low Risk': {'Goal State': 1}, 'Goal State': {}}
path_b, cost_b = a_star_search(graph_b, 'Low Risk', 'Goal State', heuristic)
facts_b  = generate_initial_facts(**{k:v for k,v in case_b.items()})
final_b, trace_b = forward_chaining(facts_b, knowledge_base)

# ── Print Comparison ──────────────────────────────────────────────────────────
print("=" * 65)
print("  CASE A — HIGH RISK PATIENT (Age 63, Multiple Risk Factors)")
print("=" * 65)
print(f"  A* Path  : {' → '.join(path_a)}")
print(f"  Cost     : {cost_a}  |  Steps: {len(path_a)}")
print(f"  KB Rules Fired   : {len(trace_a)}")
print(f"  New Facts Derived: {len(final_a - facts_a)}")

print("\n" + "=" * 65)
print("  CASE B — LOW RISK PATIENT (Age 38, Minimal Risk Factors)")
print("=" * 65)
print(f"  A* Path  : {' → '.join(path_b)}")
print(f"  Cost     : {cost_b}  |  Steps: {len(path_b)}")
print(f"  KB Rules Fired   : {len(trace_b)}")
print(f"  New Facts Derived: {len(final_b - facts_b)}")

print("\n" + "=" * 65)
print("  COMPARISON TABLE")
print("=" * 65)
print(f"  {'Metric':<35} {'Case A (High)':>13} {'Case B (Low)':>12}")
print(f"  {'-'*62}")
print(f"  {'ML Risk State':<35} {'High Risk':>13} {'Low Risk':>12}")
print(f"  {'A* Path Steps':<35} {len(path_a):>13} {len(path_b):>12}")
print(f"  {'A* Total Cost':<35} {cost_a:>13} {cost_b:>12}")
print(f"  {'Initial KB Facts':<35} {len(facts_a):>13} {len(facts_b):>12}")
print(f"  {'Rules Fired':<35} {len(trace_a):>13} {len(trace_b):>12}")
print(f"  {'New Facts Derived':<35} {len(final_a-facts_a):>13} {len(final_b-facts_b):>12}")
print()
print("  KEY OBSERVATIONS:")
print("  1. Different ML probability → different initial state → different A* path")
print("  2. More initial facts → more rules fire → richer clinical conclusions")
print("  3. Both A* and KB agree — cross-validation increases clinical trust")
print("  4. Low risk patient: 1-step path. High risk: 7+ step pathway.")

---
## Section 8 — TF-IDF Medical Knowledge Search
> Build a medical document corpus and search it using TF-IDF + Cosine Similarity.  
> This is the NLP component — retrieves supporting evidence for clinical decisions.

In [ ]:
# ── Medical Knowledge Corpus ──────────────────────────────────────────────────
documents = [
    {"title": "Heart Disease",
     "text": "Heart disease describes conditions affecting the heart and blood vessels. Common causes include high blood pressure, high cholesterol, smoking and diabetes. (Source: American Heart Association)"},
    {"title": "Blood Pressure",
     "text": "High blood pressure (hypertension) forces the heart to work harder and increases cardiovascular risk. A systolic BP >= 140 mmHg is Hypertension Stage 2. (Source: AHA, CDC)"},
    {"title": "Cholesterol",
     "text": "High cholesterol causes plaque formation inside arteries, narrowing them and increasing coronary artery disease risk. Total Cholesterol >= 240 mg/dL is considered high. (Source: NHLBI, AHA)"},
    {"title": "ECG Evaluation",
     "text": "Electrocardiography (ECG) records electrical activity of the heart and detects rhythm abnormalities or muscle damage, including exercise-induced angina indicators. (Source: Mayo Clinic, NHLBI)"},
    {"title": "Exercise Angina",
     "text": "Exercise-induced angina is chest pain during physical activity, indicating myocardial ischemia due to reduced blood flow from coronary artery disease. (Source: Mayo Clinic, NHLBI)"},
    {"title": "ST Depression",
     "text": "Oldpeak (ST depression induced by exercise) > 2mm indicates an abnormal stress response and warrants further evaluation for myocardial ischemia. (Source: Cardiology Practice, Mayo Clinic)"},
    {"title": "Cardiology Consultation",
     "text": "Cardiology consultation is advised for patients with advanced age (>=60), existing conditions, or multiple cardiovascular risk factors. (Source: AHA, General Clinical Practice)"},
    {"title": "A* Search in Healthcare",
     "text": "A* search finds optimal clinical pathways by minimising total intervention cost. It uses an admissible heuristic to guarantee the lowest-cost path from current risk state to treatment goal."},
    {"title": "Forward Chaining Inference",
     "text": "Forward chaining applies IF-THEN rules iteratively from known facts to derive new clinical conclusions. It runs until no new facts can be inferred (fixed point). Used in clinical expert systems."},
    {"title": "XGBoost Classification",
     "text": "XGBoost is a gradient-boosted decision tree model known for high accuracy on tabular clinical data. Trained with SMOTE for class balance and threshold 0.35 to maximise recall. (Source: Chen & Guestrin 2016)"},
]

# ── Build TF-IDF Index ────────────────────────────────────────────────────────
corpus       = [doc["text"] for doc in documents]
vectorizer   = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"✅ TF-IDF index built — {len(documents)} medical documents, {tfidf_matrix.shape[1]} terms.")

In [ ]:
# ── Medical Search Function ───────────────────────────────────────────────────
def medical_search(query):
    """
    TF-IDF Search: convert query to vector, compute cosine similarity
    with all documents, return the best matching document.
    """
    query_vec  = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    best_idx   = similarity.argmax()
    return documents[best_idx], similarity[best_idx]

# ── Example Queries ───────────────────────────────────────────────────────────
queries = [
    "high blood pressure treatment",
    "ST depression exercise angina",
    "forward chaining expert system rules",
]

print("=" * 60)
print("  TF-IDF MEDICAL SEARCH — EXAMPLE QUERIES")
print("=" * 60)
for q in queries:
    result, score = medical_search(q)
    print(f"\n  Query  : '{q}'")
    print(f"  Match  : {result['title']}  (similarity={score:.3f})")
    print(f"  Answer : {result['text'][:120]}...")

In [ ]:
# ── Interactive Medical Q&A ───────────────────────────────────────────────────
# Run this cell to ask your own questions from the medical corpus.
print("Medical AI Search — type 'done' to stop")
print("-" * 50)
while True:
    question = input("\nAsk a medical question: ").strip()
    if question.lower() in ("", "done", "exit", "quit"):
        print("Search session ended.")
        break
    result, score = medical_search(question)
    print(f"\n  Topic     : {result['title']}")
    print(f"  Relevance : {score:.3f}")
    print(f"  Answer    : {result['text']}")

---
## Section 9 — Complete System Integration Summary
> Full pipeline from raw patient data → ML → A* → Knowledge Base → Recommendations.

In [ ]:
# ── Clinical Recommendations from Patient Data ───────────────────────────────
recommendations = []
if trestbps >= 140: recommendations.append("Blood pressure control recommended (AHA Stage 2 Hypertension)")
if chol >= 240:     recommendations.append("Cholesterol management recommended (NHLBI threshold)")
if exang == 1:      recommendations.append("ECG evaluation recommended (exercise-induced angina)")
if oldpeak > 2:     recommendations.append("Stress testing recommended (ST depression > 2mm)")
if age >= 60:       recommendations.append("Cardiology consultation advised (age-related risk)")

# ── Full Pipeline Summary ─────────────────────────────────────────────────────
print("="*65)
print("  CARDIOAI — COMPLETE INTEGRATED PIPELINE SUMMARY")
print("="*65)

print("\n  INPUT:")
print(f"    14 clinical features for patient (age={age}, sex={'M' if sex==1 else 'F'})")

print("\n  STEP 1 — ML MODEL (XGBoost)")
print(f"    Probability : {round(probability*100,2)}%")
print(f"    Prediction  : {'Heart Disease' if prediction==1 else 'No Heart Disease'}")
print(f"    Risk State  : {risk_state}")

print("\n  STEP 2 — AI PLANNING (A* Search)")
print(f"    Path   : {' → '.join(path) if path else 'N/A'}")
print(f"    Cost   : {cost}")

print("\n  STEP 3 — KNOWLEDGE BASE (Forward Chaining)")
print(f"    Rules fired     : {len(inference_trace)}")
print(f"    New conclusions : {len(new_facts)}")

print("\n  STEP 4 — TF-IDF MEDICAL SEARCH")
print(f"    Corpus  : {len(documents)} medical documents")
print(f"    Index   : {tfidf_matrix.shape[1]} TF-IDF terms")

print("\n  CLINICAL RECOMMENDATIONS:")
if recommendations:
    for r in recommendations:
        print(f"    • {r}")
else:
    print("    • Routine monitoring — no acute risk factors detected")

print("\n  SYSTEM VALIDATION:")
print(f"    ✅ ML Risk: {risk_state}")
print(f"    ✅ A* Path: {len(path) if path else 0} steps to goal")
print(f"    ✅ KB Inferred: {len(new_facts)} clinical conclusions")
print(f"    ✅ All components executed successfully")
print("="*65)